## 3. Compute Shade Temperature (Boston Example)

This notebook shows how temperature in shade was computed

In [1]:
import os, shutil, cv2,datetime, torch, torchvision.transforms, json, torchvision, copy, time, open_clip, sys,PIL.Image
os.chdir('/home/klimenko/CoolingMachines/explore_flir')
import matplotlib.pyplot as plt
print(os.getcwd())
import pandas as pd
import numpy as np
import matplotlib.colors as mcolors
from PIL import Image
import time
import numpy as np
from scipy.ndimage import zoom
from tqdm import tqdm
import matplotlib.pyplot as plt
from PIL import Image,ImageFile
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import torch
from torch.utils.data import random_split
import torch.nn as nn
import pandas as pd
from collections import Counter
from mit_semseg.config import cfg
from mit_semseg.dataset import TestDataset
from mit_semseg.models import ModelBuilder, SegmentationModule
from transformers import CLIPSegProcessor, CLIPSegForImageSegmentation

####################################
# ADE20K-based Scene Parsing Segmentation to Extract Facade
###################################

# This module (mit_semseg) is identical to https://pypi.org/project/mit-semseg/
# Obtainted from:  https://github.com/CSAILVision/sceneparsing

# Reference: Semantic Understanding of Scenes through ADE20K Dataset. 
# B. Zhou, H. Zhao, X. Puig, T. Xiao, S. Fidler, A. Barriuso and A. Torralba.
# International Journal on Computer Vision (IJCV), 2018. (https://arxiv.org/pdf/1608.05442.pdf)


net_encoder = ModelBuilder.build_encoder(arch='resnet50dilated',fc_dim=2048,weights='ckpt/ade20k-resnet50dilated-ppm_deepsup/encoder_epoch_20.pth')
net_decoder = ModelBuilder.build_decoder(arch='ppm_deepsup',fc_dim=2048,num_class=150,weights='ckpt/ade20k-resnet50dilated-ppm_deepsup/decoder_epoch_20.pth',use_softmax=True)
crit = torch.nn.NLLLoss(ignore_index=-1)
segmentation_module = SegmentationModule(net_encoder, net_decoder, crit)
segmentation_module.eval()
device = torch.device("cuda:0")
segmentation_module.to(device)
pil_to_tensor = torchvision.transforms.Compose([torchvision.transforms.ToTensor(),torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  ])


####################################
# ClipSeg
###################################
# Source: https://huggingface.co/docs/transformers/model_doc/clipseg
# Reference: Luddecke, Timo, and Alexander Ecker. "Image segmentation using text and image prompts."
# Proceedings of the IEEE/CVF conference on computer vision and pattern recognition. 2022.
from transformers import CLIPSegProcessor, CLIPSegForImageSegmentation
processor = CLIPSegProcessor.from_pretrained("CIDAS/clipseg-rd64-refined")
model = CLIPSegForImageSegmentation.from_pretrained("CIDAS/clipseg-rd64-refined")

Couldn't import dot_parser, loading of dot files will not be possible.


/home/klimenko/anaconda3/envs/facade/lib/python3.10/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


/home/klimenko/CoolingMachines/explore_flir
Loading weights for net_encoder
Loading weights for net_decoder


2026-01-11 20:07:07.148948: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-11 20:07:08.617047: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-11 20:07:12.009789: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/home/klimenko/anaconda3/envs/facade/lib/python3.10/site-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [3]:
def segment_image(path):
    pil_image = PIL.Image.open(path).convert('RGB')
    width, height = pil_image.size
    resized_pil_image = pil_image
    img_data = pil_to_tensor(resized_pil_image)
    singleton_batch = {'img_data': img_data[None].to(device)}
    output_size = img_data.shape[1:]
    with torch.no_grad():
        scores = segmentation_module(singleton_batch, segSize=output_size)
    _, pred = torch.max(scores, dim=1)
    pred = pred.cpu()[0].numpy()
    return pred

def obtain_temp_scale(img_fresh, scalebar):
    linear_interpolation_array_temp = np.linspace(0,40, len(scalebar)).astype(float)
    img_temp = np.interp(img_fresh[:,:,0].astype(np.float32),scalebar[:,0], linear_interpolation_array_temp)
    return img_temp

In [4]:
result = pd.read_csv('/home/klimenko/CoolingMachines/explore_flir/dataset/BOSTON_FLIR_DATASET/all.csv')

In [5]:
# Example of Using CLIPSeg to segment a tree image

from PIL import Image
import requests
prompts = ['shade', 'shadow']
import torch

image = Image.open(list(result['visual_path'])[89])
inputs = processor(text=prompts, images=[image] * len(prompts), padding="max_length", return_tensors="pt")
with torch.no_grad():
  outputs = model(**inputs)
preds = outputs.logits.unsqueeze(1)
shade_mask = np.array((torch.sigmoid(preds[0][0]) > 0.25).float())
shade_mask = cv2.resize(shade_mask.astype(np.float32), (480, 640))

In [28]:
## Definition of shade calculation
def func_shade(visual_path,thermal_path):

    pred = segment_image(visual_path)
    thermal_img_fresh = cv2.imread(thermal_path)

    file_name, file_extension = os.path.splitext(os.path.basename(thermal_path))
    
    new_file_name = file_name + file_extension.lower().replace(".jpeg", ".jpg") + ".npy"
    scalebar = np.load('dataset/BOSTON_FLIR_DATASET/all/watermarked/'+new_file_name)
    thermal = obtain_temp_scale(thermal_img_fresh, scalebar)

    b = cv2.imread(visual_path)
    b = b.mean(axis=2)
    light_threshold_mask = np.where(b[:,:]<45, 1, 0)
    
    
    # Use CLIPSeg to calculate shade region:
    image = Image.open(visual_path)
    inputs = processor(text=prompts, images=[image] * len(prompts), padding="max_length", return_tensors="pt")
    with torch.no_grad():
      outputs = model(**inputs)
    
    preds = outputs.logits.unsqueeze(1)
    shade_mask = np.array((torch.sigmoid(preds[0][0]) > 0.25).float())
    shade_mask = cv2.resize(shade_mask.astype(np.float32), (480, 640))
    
    ##################

    ### PICK LABELS 
    # Depending on the type of material that the shade falls onto, we select a material to segment
    # In this notebook, we show an example of detecting a shade falling onto a wall of a builing

    # 9 - grass
    # 6,11 - asphalt
    # 1 - wall

    ##################
    
    target_index = [1]

    other_mask = np.isin(pred, target_index).astype(np.float32)

    other_mask = cv2.resize(other_mask.astype(np.float32), (480, 640))
    light_threshold_mask = cv2.resize(light_threshold_mask.astype(np.float32), (480, 640))

    shade = other_mask * thermal[:,:] * shade_mask
    shade_temperature = np.nanmean(shade[shade > 10])
    
    # We exclude areas that are generally not too dark (not in shade) so that they do not conflict with the shade
    # of the tree
    outside = other_mask * thermal[:,:] * (1-shade_mask)*(1-light_threshold_mask)
    outside_temperature = np.nanmean(outside[outside > 10])
    
    return shade_temperature, outside_temperature

In [ ]:
# Run Shade Calculation Function throughout the dataset

import pandas as pd
import cv2

df = pd.read_csv('dataset/BOSTON_FLIR_DATASET/all.csv')

# Initialize empty lists to hold the computed values
tree_temperature_values = []
other_temperature_values = []

# Loop over the rows in the DataFrame and apply the function
total_rows = len(df)
for idx, row in df.iterrows():
    
    try:
        # Extract values using the image path and info

        tree_temperature, other_temperature = func_shade(row['visual_path'], row['thermal_path'])
        print(tree_temperature, other_temperature)

        # Append the results to the respective lists
        tree_temperature_values.append(tree_temperature)
        other_temperature_values.append(other_temperature)

        # Print progress manually
        if (idx + 1) % 10 == 0 or (idx + 1) == total_rows:
            print(f"Processed {idx + 1}/{total_rows} rows____________________")
            
    except:
        print('F')
        tree_temperature_values.append(np.nan)
        other_temperature_values.append(np.nan)

# Add the computed values as new columns in the DataFrame
df['shade_wall'] = tree_temperature_values
df['outside_wall'] = other_temperature_values


34.193351277019644 35.56316383882764
27.26716941528997 36.65503105663688
31.496993494344306 35.90156845109827
29.442418730477623 36.282745812518584
34.17446755393677 37.84800026277575
33.42577598720293 36.22035260387849
26.948615112370884 35.91403442690354
34.359351522966755 34.117236421623055
33.56523582242196 35.20883860403072
39.087370631071764 35.26009459043303
Processed 10/497 rows____________________


/tmp/ipykernel_10693/624709606.py:54: RuntimeWarning: Mean of empty slice
  shade_temperature = np.nanmean(shade[shade > 10])


nan 35.51905882815801
nan 32.47095927445115
15.988251727175344 23.47955922713956
31.736211871209914 37.09243023920897
nan 27.610487843344863
35.16860193721555 21.808593214542608
29.220571364728308 37.223300764812016
25.958367941521352 27.39302579436629
28.846626072346577 36.12536805546895
nan 26.982140713370626
Processed 20/497 rows____________________
34.322876372048384 36.90326175276474
nan 36.347542216777256
25.62943938944808 27.97519996183187
31.43130647511481 37.29374084694118
nan 26.272941311834057
nan 37.054780642953425
nan 30.332653713030673
37.59961653565617 36.110936547560186
nan 29.569825877123915
34.21925528855313 37.80336304086223
Processed 30/497 rows____________________
31.29879736889849 38.106150269723145
nan 30.90618700244303
nan 29.763989297129694
36.33325740968182 38.05486090341023
nan 33.59569831895912
nan 34.01108198656572
nan 34.79620749792664
nan 39.474677847029135
nan 39.07003542527465
nan 39.05812606113604
Processed 40/497 rows____________________
22.0071848010

In [ ]:
df.to_csv('dataset/BOSTON_FLIR_DATASET/all.csv')